In [1]:
import retrieve_utils

print(retrieve_utils.__file__)

print(dir(retrieve_utils))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

C:\Users\Admin\retrieve_utils.py
['BM25Okapi', 'CrossEncoder', 'SentenceTransformer', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'bm25', 'build_context', 'chromadb', 'client', 'collection', 'embedding_model', 'hybrid_search', 'lexical_dataset', 'lexical_search', 'np', 'pd', 'remove_duplicates', 'rerank_results', 'reranker', 'retrieve_context', 'semantic_search', 'top_k']


In [2]:
import importlib
import retrieve_utils

importlib.reload(retrieve_utils)

from retrieve_utils import retrieve_context

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [3]:
import inspect
import retrieve_utils

print(inspect.getsource(retrieve_utils.semantic_search))

def semantic_search(query, top_k=5):

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=[
            "documents",
            "metadatas",
            "distances"
        ],
    )

    return results



In [4]:
import inspect
import retrieve_utils

print(inspect.getsource(retrieve_utils.lexical_search))

def lexical_search(query, top_k=5):

    query_tokens = query.lower().split()
    scores = bm25.get_scores(query_tokens)
    top_indices = scores.argsort()[-top_k:][::-1]
    results = lexical_dataset.iloc[top_indices].copy()
    results["bm25_score"] = scores[top_indices]

    return results



In [5]:
import importlib
import retrieve_utils

importlib.reload(retrieve_utils)

print(inspect.getsource(retrieve_utils.lexical_search))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

def lexical_search(query, top_k=5):

    query_tokens = query.lower().split()
    scores = bm25.get_scores(query_tokens)
    top_indices = scores.argsort()[-top_k:][::-1]
    results = lexical_dataset.iloc[top_indices].copy()
    results["bm25_score"] = scores[top_indices]

    return results



In [13]:
# =============================================================================
# Import Required Libraries
# =============================================================================
# Data manipulation and loading datasets
import pandas as pd

# Google AI Studio client for interacting with Large Language Models (LLMs)
# Google Gemini API
from google import genai

In [14]:
# ==========================================================
# Import Retrieval Pipeline
# ==========================================================
from retrieve_utils import retrieve_context

print("Retrieve module loaded successfully.")

Retrieve module loaded successfully.


In [40]:
# =============================================================================
# User Question
# =============================================================================
query = input("Ask your question: ")

Ask your question:  When folded in the sitting position, how high off the ground is the seat?


In [41]:
# =============================================================================
# Retrieve Relevant Context
# Generate the final context using the complete RAG retrieval pipeline
# (Hybrid Search → Reranking → Duplicate Removal → Context Building)
# =============================================================================
context, retrieved_chunks = retrieve_context(query)

In [42]:
# =============================================================================
# Prompt Template
# =============================================================================
prompt = f"""
You are an expert Amazon Product Question Answering Assistant using a ReAct-style approach.

Your goal is to answer customer questions accurately using the retrieved product context.

Follow this process internally:

1. Analyze:
- Understand what the customer is asking.
- Identify the required information (fit, size, compatibility, capacity, usage, etc.).

2. Retrieve Evidence:
- Examine the provided context.
- Select only information relevant to the question.
- Compare multiple reviews if available.

3. Decide:
- Determine the best answer based on the available evidence.
- If information is conflicting, explain the uncertainty.
- If information is missing, do not guess.

4. Respond:
- Provide a clear and concise customer-friendly answer.
- Rewrite the information naturally.
- Do not copy the context word-for-word.
- Do not mention this reasoning process.

Rules:
- Use ONLY the provided context.
- Never invent product specifications.
- If the context does not provide enough evidence, answer:
  "I don't have enough information from the available product reviews to answer this question."

Retrieved Context:
{context}

Customer Question:
{query}

Final Answer:
"""

In [43]:
print(context)

Context 1
Question: when folded in the sitting position, how high off the ground is the seat? Answer: it is nine inches front to back, and nine inches at it widest point side to side in the front. it tapers to seven inches side to side in the back.

Context 2
Question: when folded in the sitting position, how high off the ground is the seat? Answer: it is 30 inches above the ground....and i still love mine...especially since it's time to do a little gardening!

Context 3
Question: when folded in the sitting position, how high off the ground is the seat? Answer: it is roughly 9 inches front to back and 8 inches side to side. here is a tip for those who are as slow on the uptake as i. when i first used it, i thought the handle support wassupposed to be the back. found it to be uncomfortable and leaning too far back for any back support - also was not very sturdy to sit on. duh! my grandson pointed out that the seat is designed to be straddled facing the handle, using it as arm support. w

In [44]:
# =============================================================================
# Generate Response
# =============================================================================
client = genai.Client(api_key="api_key")

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt)

answer = response.text

print(answer)

When folded in the sitting position, the seat is 30 inches above the ground.
